In [1]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [2]:
(train_input, train_target), (test_input, test_target) \
    = tf.keras.datasets.fashion_mnist.load_data()

In [3]:
# 정규화
train_scaled = train_input / 255.0  # / 255.0 -> 0, 1사이의 값으로 매칭되게 변환
test_scaled = test_input / 255.0

In [4]:
train_scaled, val_scaled, train_target, val_target = \
    train_test_split(train_scaled, train_target, \
                     test_size=0.2, random_state=42)

In [5]:
model = tf.keras.Sequential()

model.add(tf.keras.Input(shape=(28, 28)))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(units=1024, activation='relu'))
model.add(tf.keras.layers.Dense(units=512, activation='relu'))
model.add(tf.keras.layers.Dense(units=10, activation='softmax'))

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │       803,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,333,770 (5.09 MB)

 Trainable params: 1,333,770 (5.09 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# SGD 단점 개선 : Hyper 파라미터 Momentum 적용
# 모멘텀에 의한 최적화 갱신 경로
sgd = tf.keras.optimizers.SGD(momentum=0.9)

# model 연결
model.compile(optimizer=sgd,
              loss='sparse_categorical_crossentropy',  # 모델에게 onehot encoding까지 실행하면서 수행해!
              metrics=['accuracy'])

# 학습
model.fit(train_scaled, train_target, epochs=50)

Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8157 - loss: 0.5164
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8609 - loss: 0.3771
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8756 - loss: 0.3373
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8860 - loss: 0.3088
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8914 - loss: 0.2888
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8971 - loss: 0.2727
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9028 - loss: 0.2582
Epoch 8/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9086 - loss: 0.2463
Epoch 9/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9100 - loss: 0.2354
Epoch 10/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9143 - loss: 0.2268
Epoch 11/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9178 - loss: 0.2156
Epoch 12/50
1500/1500 ━━━━━━━━

In [8]:
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8989 - loss: 0.4524


[0.45239225029945374, 0.8989166617393494]

In [ ]:
# Adagrad 알고리즘 적용
# 학습률 값이 너무 작으면 학습 시간이 너무 길어지고, 반대로 너무 크면 발산하여 학습이 제대로 이뤄지지 않음
# SGD 단점 개선
adagrad = tf.keras.optimizers.Adagrad()

model.compile(optimizer=adagrad,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_scaled, train_target, epochs=50)

In [ ]:
# Adagrad 알고리즘 적용, 정확도 90% 이상
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9068 - loss: 0.4957


[0.4956769645214081, 0.9067500233650208]

In [11]:
# Adam 알고리즘
# 모멘텀과 AdaGrad를 융합한 듯한 방법으로 2015년에 제안된 새로운 방법
# 속도 고려, learning rate까지 고려

adam = tf.keras.optimizers.Adam()
model.compile(optimizer=adam,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_scaled, train_target, epochs=50)

Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8993 - loss: 0.2848
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9105 - loss: 0.2445
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9146 - loss: 0.2308
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9171 - loss: 0.2203
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9212 - loss: 0.2083
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9254 - loss: 0.1982
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9286 - loss: 0.1883
Epoch 8/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9325 - loss: 0.1770
Epoch 9/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9346 - loss: 0.1754
Epoch 10/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9367 - loss: 0.1648
Epoch 11/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9398 - loss: 0.1585
Epoch 12/50
1500/1500 ━━━━━━━━

In [ ]:
model.evaluate(val_scaled, val_target)  # 학습에 한번도 사용하지 않은 검증데이터인 경우 83%

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8897 - loss: 0.7469


[0.746913492679596, 0.8896666765213013]